# GuardianAire Thermal Person Detection
Fine-tune YOLO11-N on UAV thermal imagery for person detection.

**Runtime:** Go to Runtime → Change runtime type → **T4 GPU**

**Time estimate:** ~45 min for 100 epochs on T4

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q ultralytics gdown

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download HIT-UAV Dataset
High-altitude infrared thermal dataset from UAV perspective.
2,898 thermal images with Person/Car/Bicycle annotations.

In [ ]:
import os
from pathlib import Path

DATASET_DIR = Path("/content/datasets/hit-uav")

if not DATASET_DIR.exists():
    print("Downloading HIT-UAV dataset...")
    !git clone --depth 1 https://github.com/suojeong/HIT-UAV.git {DATASET_DIR}/raw
    print("Done!")
else:
    print(f"Dataset already exists at {DATASET_DIR}")

# Verify dataset structure
for split in ['train', 'val', 'test']:
    img_dir = DATASET_DIR / 'raw' / 'normal' / 'images' / split
    lbl_dir = DATASET_DIR / 'raw' / 'normal' / 'labels' / split
    n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*'))) if lbl_dir.exists() else 0
    print(f"  {split}: {n_imgs} images, {n_lbls} labels")

In [ ]:
# Create dataset YAML config
dataset_yaml = """
path: /content/datasets/hit-uav/raw/normal
train: images/train
val: images/val
test: images/test

names:
  0: Person
  1: Car
  2: Bicycle
"""

with open('/content/hit_uav.yaml', 'w') as f:
    f.write(dataset_yaml)

print("Dataset config written to /content/hit_uav.yaml")

## 3. Preview Sample Images

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Show a few sample thermal images
train_imgs = sorted((DATASET_DIR / 'raw' / 'normal' / 'images' / 'train').glob('*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, train_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('HIT-UAV Thermal Training Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Train YOLO11-N
Fine-tune the nano model with thermal-optimized augmentation.
~45 minutes on T4 GPU for 100 epochs.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11 nano
model = YOLO('yolo11n.pt')

# Fine-tune on thermal dataset
results = model.train(
    data='/content/hit_uav.yaml',
    epochs=100,
    imgsz=640,
    batch=-1,          # Auto batch size for GPU memory
    device=0,          # GPU
    project='/content/runs/thermal',
    name='train_yolo11n',
    # Thermal-optimized augmentation
    hsv_h=0.0,         # No hue (thermal is grayscale)
    hsv_s=0.0,         # No saturation
    hsv_v=0.3,         # Brightness variation
    degrees=10.0,      # Slight rotation
    translate=0.1,
    scale=0.5,         # Altitude variation
    fliplr=0.5,
    flipud=0.1,        # Aerial view
    mosaic=0.8,
    mixup=0.1,
    patience=20,       # Early stopping
    save=True,
    plots=True,
)

## 5. Evaluate Results

In [ ]:
# Load best model
best_model = YOLO('/content/runs/thermal/train_yolo11n/weights/best.pt')

# Validate on test set
metrics = best_model.val(
    data='/content/hit_uav.yaml',
    split='test',
    plots=True,
)

print(f"\n=== Test Results ===")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Visualize predictions on test images
test_imgs = sorted((DATASET_DIR / 'raw' / 'normal' / 'images' / 'test').glob('*'))[:8]

results = best_model.predict(
    source=[str(p) for p in test_imgs],
    imgsz=640,
    conf=0.25,
    save=True,
    project='/content/runs/thermal',
    name='test_predictions',
)

# Display predictions
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
pred_dir = Path('/content/runs/thermal/test_predictions')
pred_imgs = sorted(pred_dir.glob('*'))[:8]

for ax, img_path in zip(axes.flat, pred_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('Thermal Person Detection — Test Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Training Curves

In [ ]:
from IPython.display import Image, display

# Show training curves
results_img = Path('/content/runs/thermal/train_yolo11n/results.png')
if results_img.exists():
    display(Image(filename=str(results_img), width=900))

# Show confusion matrix
cm_img = Path('/content/runs/thermal/train_yolo11n/confusion_matrix.png')
if cm_img.exists():
    display(Image(filename=str(cm_img), width=600))

## 7. Export to TFLite for Android

In [ ]:
# Export to TFLite (FP16)
tflite_path = best_model.export(
    format='tflite',
    imgsz=640,
    half=True,  # FP16 quantization
)

tflite_file = Path(tflite_path)
size_mb = tflite_file.stat().st_size / (1024 * 1024)
print(f"\nTFLite model: {tflite_path}")
print(f"Size: {size_mb:.1f} MB")

# Also export INT8 for comparison
tflite_int8_path = best_model.export(
    format='tflite',
    imgsz=640,
    int8=True,
)

tflite_int8_file = Path(tflite_int8_path)
size_int8_mb = tflite_int8_file.stat().st_size / (1024 * 1024)
print(f"\nTFLite INT8 model: {tflite_int8_path}")
print(f"Size: {size_int8_mb:.1f} MB")

## 8. Download Models
Download the trained models to your local machine.

In [ ]:
from google.colab import files
import shutil

# Package models for download
export_dir = Path('/content/export')
export_dir.mkdir(exist_ok=True)

# Copy key files
shutil.copy('/content/runs/thermal/train_yolo11n/weights/best.pt', export_dir / 'thermal_person_best.pt')
if tflite_file.exists():
    shutil.copy(tflite_file, export_dir / 'thermal_person_fp16.tflite')
if tflite_int8_file.exists():
    shutil.copy(tflite_int8_file, export_dir / 'thermal_person_int8.tflite')

# Create zip
shutil.make_archive('/content/thermal_person_models', 'zip', export_dir)
print("Models packaged!")

# Download
files.download('/content/thermal_person_models.zip')

## 9. Quick Benchmark
Test inference speed to estimate Android performance.

In [ ]:
import time

# Benchmark PyTorch model on GPU
test_img = str(test_imgs[0])

# Warmup
for _ in range(5):
    best_model.predict(test_img, imgsz=640, verbose=False)

# Benchmark
times = []
for _ in range(50):
    t0 = time.perf_counter()
    best_model.predict(test_img, imgsz=640, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

avg_ms = sum(times) / len(times)
print(f"GPU inference: {avg_ms:.1f} ms/frame ({1000/avg_ms:.0f} FPS)")
print(f"\nNote: Android tablet inference will be ~3-10x slower than T4 GPU")
print(f"Estimated Android: {avg_ms * 5:.0f}-{avg_ms * 10:.0f} ms/frame ({1000/(avg_ms*10):.0f}-{1000/(avg_ms*5):.0f} FPS)")

---
## Next Steps

1. **Download** the exported models (cell above)
2. **Copy TFLite** to `autel-mobilesdk-2.0/app/src/main/assets/`
3. **Integrate** TFLite inference into the Android app
4. **Collect** your own thermal data from drone flights for fine-tuning
5. **Iterate** — retrain with your custom data + HIT-UAV combined